# Pipeline Runner

Run the full galaxy classification pipeline from this notebook — one cell per stage.

**Each stage is independent.** Already-complete stages are skipped automatically (their output files exist).  
Set `FORCE = True` in the Config cell to re-run everything from scratch.

| Stage | What it does | Time | Device |
|---|---|---|---|
| 1. split-full | Full-data 85/15 split (keeps test set fixed) | ~1 min | CPU |
| 2. tabular | XGBoost + grid search | ~20 min | CPU |
| 3. image A–D | ResNet-18 configs (4 runs) | ~4–8 hr | GPU |
| 4. spectral A–D | 1D-CNN configs (4 runs) | ~2 hr | GPU |
| 5. fusion | Late fusion (best encoders) | ~1 hr | GPU |
| 6. ablation | Fusion ×3 (no-image / no-spec / no-tab) | ~3 hr | GPU |
| 7. interpret | Feature importance + GradCAM | ~5 min | CPU/GPU |
| 8. physical | Star-forming vs passive | ~30 min | GPU |
| 9. evaluate | Final summary table | ~1 min | CPU |

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import subprocess, sys, time
from pathlib import Path

FORCE = False   # set True to re-run even if output already exists
ROOT  = Path("..")  # repository root (notebook is in notebooks/)

def run(module, *args, cwd=ROOT):
    """Run `uv run python -m <module> [args]` and stream output."""
    cmd = ["uv", "run", "python", "-m", module, *args]
    print(f">>> {' '.join(cmd)}\n")
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(cwd),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"Stage failed (exit {proc.returncode}) after {elapsed:.0f}s")
    print(f"\n✓ Done in {elapsed:.0f}s")

def skip_if_exists(*paths, name=""):
    """Return True (skip) if all sentinel files exist and FORCE is False."""
    if FORCE:
        return False
    if all((ROOT / p).exists() for p in paths):
        print(f"[skip] {name or paths[0]} already exists")
        return True
    return False

print("Config loaded. FORCE =", FORCE)

In [ ]:
if not skip_if_exists(
    "input/tables/split_train.csv",
    "input/tables/split_val.csv",
    "input/tables/split_test.csv",
    name="split",
):
    run("src.data.split")

In [ ]:
if not skip_if_exists(
    "input/tables/split_train_full.csv",
    "input/tables/split_val_full.csv",
    name="split-full",
):
    run("src.data.split", "--full-data")

---
## Stage 1 — Full-data split
Expands train/val to all 214k labeled rows; keeps `split_test.csv` unchanged.

In [ ]:
if not skip_if_exists(
    "input/tables/split_train_full.csv",
    "input/tables/split_val_full.csv",
    name="split-full",
):
    run("src.data.split")

---
## Stage 2 — Tabular baseline (XGBoost)
Grid search (27 combos, 5-fold CV) then retrain best config on full 154k training set.

In [ ]:
if not skip_if_exists(
    "checkpoints/tabular/xgb_model_full.pkl",
    "checkpoints/tabular/test_results_full.pkl",
    name="tabular",
):
    run("src.train.tabular", "--grid-search")

---
## Stage 3 — Image baseline (ResNet-18)
Four configs. **GPU required.** Best is Config C (dropout=0.3).

In [ ]:
# Config C — dropout=0.3  ← BEST
if not skip_if_exists("checkpoints/image/resnet18_configC_best.pth", name="image Config C"):
    run("src.train.image", "--dropout", "0.3", "--suffix", "configC")

In [ ]:
# Config D — no ImageNet pretraining
if not skip_if_exists("checkpoints/image/test_results_configD.pkl", name="image Config D"):
    run("src.train.image", "--no-pretrain", "--dropout", "0.3", "--suffix", "configD")

---
## Stage 5 — Spectral baseline (1D-CNN)
Four configs. Config A (baseline) is best.

In [ ]:
# Config A — baseline
if not skip_if_exists("checkpoints/spectral/cnn1d_best.pth", name="spectral Config A"):
    run("src.train.spectral")

---
## Stage 4 — Spectral baseline (1D-CNN)
Four configs. Config A (baseline) is best.

In [ ]:
# Config C — wider filters (base_filters=64)
if not skip_if_exists("checkpoints/spectral/test_results_configC.pkl", name="spectral Config C"):
    run("src.train.spectral", "--base-filters", "64", "--suffix", "configC")

In [ ]:
# Config D — dropout=0.3
if not skip_if_exists("checkpoints/spectral/test_results_configD.pkl", name="spectral Config D"):
    run("src.train.spectral", "--dropout", "0.3", "--suffix", "configD")

---
## Stage 6 — Late fusion
Uses best encoders: ResNet Config C + 1D-CNN Config A. Two-phase training.

In [ ]:
if not skip_if_exists("checkpoints/fusion/test_results_bestenc.pkl", name="fusion"):
    run(
        "src.train.fusion",
        "--img-ckpt",  "checkpoints/image/resnet18_configC_best.pth",
        "--spec-ckpt", "checkpoints/spectral/cnn1d_best.pth",
        "--suffix",    "_bestenc",
    )

---
## Stage 5 — Late fusion
Uses best encoders: ResNet Config C + 1D-CNN Config A. Two-phase training.

In [ ]:
# No spectral
if not skip_if_exists("checkpoints/fusion/test_results_ablate_nospec.pkl", name="ablation no-spectral"):
    run(
        "src.train.fusion",
        "--img-ckpt",  "checkpoints/image/resnet18_configC_best.pth",
        "--spec-ckpt", "checkpoints/spectral/cnn1d_best.pth",
        "--ablate",    "spectral",
        "--suffix",    "_ablate_nospec",
    )

---
## Stage 6 — Ablation study
Three fusion variants: each with one modality zeroed during training AND evaluation.

In [ ]:
# No tabular
if not skip_if_exists("checkpoints/fusion/test_results_ablate_notab.pkl", name="ablation no-tabular"):
    run(
        "src.train.fusion",
        "--img-ckpt",  "checkpoints/image/resnet18_configC_best.pth",
        "--spec-ckpt", "checkpoints/spectral/cnn1d_best.pth",
        "--ablate",    "tabular",
        "--suffix",    "_ablate_notab",
    )

---
## Stage 8 — Interpretability
XGBoost feature importance (gain) + GradCAM activation maps on ResNet-18.

In [ ]:
if not skip_if_exists(
    "checkpoints/tabular/feature_importance.csv",
    "checkpoints/image/gradcam_samples.png",
    name="interpret",
):
    run("src.interpret")

---
## Stage 7 — Interpretability
XGBoost feature importance (gain) + GradCAM activation maps on ResNet-18.

In [ ]:
if not skip_if_exists(
    "checkpoints/physical/tabular_results.pkl",
    "checkpoints/physical/spectral_results.pkl",
    name="physical",
):
    run("src.train.physical")

---
## Stage 8 — Physical classification (stretch task)
Star-forming vs passive: tabular XGBoost + 1D-CNN spectral on 2-class labels.

In [ ]:
if not skip_if_exists("checkpoints/results_summary.csv", name="evaluate") or FORCE:
    run("src.evaluate")

---
## Stage 9 — Final evaluation summary

In [ ]:
import pandas as pd
summary = pd.read_csv(ROOT / "checkpoints/results_summary.csv")
print(summary.to_string(index=False))